In [12]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

In [13]:
# ── CONFIGURAÇÕES ──────────────────────────────────────────

REGIONS    = ['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste']
CATEGORIES = ['Notebooks', 'Periféricos', 'Smartphones', 'Acessórios', 'Monitores']

PRODUCTS = {
    'Notebooks':    ['Dell Inspiron', 'Lenovo IdeaPad', 'HP Pavilion', 'Asus VivoBook'],
    'Periféricos':  ['Mouse Logitech', 'Teclado Redragon', 'Headset HyperX', 'Webcam Logitech'],
    'Smartphones':  ['Samsung A54', 'Xiaomi 12', 'iPhone 13', 'Motorola Edge'],
    'Acessórios':   ['Cabo USB-C', 'Carregador 65W', 'Suporte Notebook', 'Hub USB'],
    'Monitores':    ['LG 24"', 'Dell 27"', 'Samsung 32"', 'AOC 24"'],
}

BASE_PRICE = {
    'Notebooks':    2800,
    'Periféricos':  180,
    'Smartphones':  1900,
    'Acessórios':   90,
    'Monitores':    1100,
}

COST_RATIO = {
    'Notebooks':    0.72,
    'Periféricos':  0.85,  # margem baixa — intencional
    'Smartphones':  0.74,
    'Acessórios':   0.65,
    'Monitores':    0.70,
}

GOALS = {
    'Sul':          320000,
    'Sudeste':      500000,
    'Centro-Oeste': 280000,
    'Norte':        180000,  # vai ficar abaixo — intencional
    'Nordeste':     210000,
}


In [14]:
# ── SAZONALIDADE ───────────────────────────────────────────
# Meses com mais venda (Black Friday, início de ano)

SEASONALITY = {
    1:  1.10,  # janeiro — volta às aulas
    2:  0.90,
    3:  0.95,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.05,  # férias
    8:  0.95,
    9:  0.95,
    10: 1.00,
    11: 1.35,  # Black Friday
    12: 1.20,  # natal
}


In [15]:
# ── DIM_SELLER ─────────────────────────────────────────────

sellers = []
seller_id = 1

for region in REGIONS:
    n_sellers = 6 if region == 'Sudeste' else 4
    for _ in range(n_sellers):
        seniority = round(random.uniform(0.5, 8.0), 1)
        sellers.append({
    'vendedor_id':       seller_id,
    'nome_vendedor':     fake.name(),
    'regiao':            region,
    'anos_experiencia':  seniority,
    'equipe':            'Comercial',
})
        seller_id += 1

dim_seller = pd.DataFrame(sellers)

In [18]:
# ── FACT_SALES ─────────────────────────────────────────────

records = []
sale_id = 1

# 2 anos de dados
dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')

for date in dates:
    month       = date.month
    seasonality = SEASONALITY[month]

    for _, seller in dim_seller.iterrows():

        # Volume base por perfil de vendedor
        if seller['regiao'] == 'Norte':
            # Norte com baixa performance — intencional
            weights = [0.55, 0.30, 0.10, 0.05]

        elif seller['anos_experiencia'] > 5:
            # Sênior vende mais
            weights = [0.15, 0.35, 0.30, 0.20]

        elif seller['anos_experiencia'] > 2:
            # Pleno
            weights = [0.25, 0.40, 0.25, 0.10]

        else:
            # Júnior
            weights = [0.40, 0.40, 0.15, 0.05]

        # Aplica sazonalidade no peso de 0 vendas
        adjusted_zero = max(0.05, weights[0] / seasonality)
        total_rest    = 1 - adjusted_zero
        scale         = total_rest / sum(weights[1:])
        adj_weights   = [adjusted_zero] + [w * scale for w in weights[1:]]

        n_sales = random.choices([0, 1, 2, 3], weights=adj_weights)[0]

        for _ in range(n_sales):
            category     = random.choice(CATEGORIES)
            product_name = random.choice(PRODUCTS[category])
            base_price   = BASE_PRICE[category]
            unit_price   = round(base_price * random.uniform(0.90, 1.15), 2)

            # Periféricos com desconto alto — intencional
            if category == 'Periféricos':
                discount_pct = round(random.uniform(0.18, 0.32), 2)
            else:
                discount_pct = round(random.uniform(0.02, 0.12), 2)

            quantity   = random.randint(1, 8)
            revenue    = round(unit_price * quantity * (1 - discount_pct), 2)
            cost       = round(unit_price * quantity * COST_RATIO[category], 2)
            profit     = round(revenue - cost, 2)
            goal_month = GOALS[seller['regiao']]

            records.append({
    'venda_id':          sale_id,
    'data':              date.strftime('%Y-%m-%d'),
    'vendedor_id':       seller['vendedor_id'],
    'regiao':            seller['regiao'],
    'categoria':         category,
    'produto':           product_name,
    'quantidade':        quantity,
    'preco_unitario':    unit_price,
    'desconto_pct':      discount_pct,
    'receita':           revenue,
    'custo':             cost,
    'lucro':             profit,
    'meta_mensal':       goal_month,
})

            sale_id += 1

fact_sales = pd.DataFrame(records)

In [21]:
# ── VALIDAÇÃO ──────────────────────────────────────────────

print("=" * 45)
print("✅ Dataset gerado com sucesso!")
print("=" * 45)
print(f"\n📋 fact_sales : {len(fact_sales):>8,} linhas")
print(f"👤 dim_seller : {len(dim_seller):>8,} linhas")
print(f"\n📅 Período    : 2022-01-01 → 2023-12-31")
print(f"🗂️  Categorias : {fact_sales['categoria'].nunique()}")
print(f"🌎 Regiões    : {fact_sales['regiao'].nunique()}")
print(f"👔 Vendedores : {fact_sales['vendedor_id'].nunique()}")

print("\n── Receita por Região ──────────────────────")
region_summary = (
    fact_sales.groupby('regiao')['receita']
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(region_summary.to_string())

print("\n── Margem por Categoria ────────────────────")
margin_summary = (
    fact_sales.groupby('categoria')
    .apply(lambda x: round((x['lucro'].sum() / x['receita'].sum()) * 100, 1))
    .sort_values()
)
for cat, margin in margin_summary.items():
    print(f"  {cat:<15} {margin}%")

✅ Dataset gerado com sucesso!

📋 fact_sales :   17,940 linhas
👤 dim_seller :       22 linhas

📅 Período    : 2022-01-01 → 2023-12-31
🗂️  Categorias : 5
🌎 Regiões    : 5
👔 Vendedores : 22

── Receita por Região ──────────────────────
regiao
Sudeste         R$ 28,225,055
Nordeste        R$ 20,965,723
Sul             R$ 17,773,408
Centro-Oeste    R$ 16,012,657
Norte            R$ 9,912,435

── Margem por Categoria ────────────────────
  Periféricos     -13.3%
  Smartphones     20.5%
  Notebooks       22.6%
  Monitores       24.7%
  Acessórios      30.1%


In [22]:
# ── EXPORTAR ───────────────────────────────────────────────
data_dir = os.path.join(os.getcwd(), '..', 'data')
os.makedirs(data_dir, exist_ok=True)

fact_path   = os.path.join(data_dir, 'fact_sales.csv')
seller_path = os.path.join(data_dir, 'dim_seller.csv')

fact_sales.to_csv(fact_path,   index=False, encoding='utf-8-sig')
dim_seller.to_csv(seller_path, index=False, encoding='utf-8-sig')

print(f"\n📁 fact_sales salvo em : {fact_path}")
print(f"📁 dim_seller salvo em : {seller_path}")
print("=" * 45)


📁 fact_sales salvo em : C:\Users\allis\Documents\Vortex Projeto\python\..\data\fact_sales.csv
📁 dim_seller salvo em : C:\Users\allis\Documents\Vortex Projeto\python\..\data\dim_seller.csv
